In [1]:
import xarray as xr
import pandas as pd

In [2]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/3D_data"
)
repeats = 100

In [3]:
# Choose year 1996
data_1996 = data.sel(time=slice("1996-01-01", "1996-12-31"))

# Print first and last time steps
print(data_1996.time[0].values, data_1996.time[-1].values)

print(data_1996)


1996-01-03 12:00:00 1996-12-28 12:00:00
<xarray.Dataset>
Dimensions:        (time: 73, y: 180, x: 360)
Coordinates:
  * time           (time) object 1996-01-03 12:00:00 ... 1996-12-28 12:00:00
  * x              (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y              (y) float64 -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
Data variables: (12/80)
    hfds           (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_0       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_1       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_10      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_11      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_12      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...             ...
    vo_lev_5       (time, y, x) float64 das

In [4]:
# Generate a new time coordinate for repeats years, repeating 1996
new_time = pd.date_range(start=str(data_1996.time[0].values), periods=repeats * len(data_1996.time), freq="5D")

In [5]:
# Assert first year are the same dates 
for i in range(len(data_1996.time)):
    print(i , end=" ")
    assert str(new_time[i]) == str(data_1996.time[i].values)

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 

In [6]:
# Repeat the data for each day to match the new time length
repeated_data = xr.concat([data_1996] * repeats, dim="time")
repeated_data['time'] = new_time  # Update the time coordinate

In [8]:
from dask.diagnostics import ProgressBar

In [7]:
# Save data
with ProgressBar():
    repeated_data.to_zarr("/pscratch/sd/s/suryad/data/3D_data_100_years")

In [10]:
with ProgressBar():
    ds_mean = data_1996.mean().compute()
    ds_mean.to_zarr("/pscratch/sd/s/suryad/data/3D_data_100_years_means")

[########################################] | 100% Completed | 10.10 s


In [11]:
with ProgressBar():
    ds_mean = data_1996.std().compute()
    ds_mean.to_zarr("/pscratch/sd/s/suryad/data/3D_data_100_years_stds")

[########################################] | 100% Completed | 11.86 s
